# Experimentation notebook
Exploration of the nail-image KNN pipeline. **This notebook deliberately never touches the test split** - the test set is used once, in `src/train_model.py`.

Run from the `notebooks/` folder with the project venv activated.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from src import config, dataset_loader, preprocessing
from src.feature_extraction import extract_from_path, extract_from_image, FEATURE_NAMES
from src.train_model import split_data
from src.evaluate_model import compute_metrics

## 1. Dataset audit

In [ ]:
df, audit = dataset_loader.load_clean_dataset(save_reports=False)
audit

In [ ]:
df['class_name'].value_counts().plot(kind='bar', color=['#0d9488', '#3b82f6'])
plt.title('Images per class (after cleaning)'); plt.show()

## 2. Look at preprocessing on a few images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, (_, r) in zip(axes, df.groupby('class_name').head(1).iterrows()):
    st = preprocessing.preprocess(preprocessing.read_image_path(r['path']))
    imgs = [('resized', st['resized'][..., ::-1]), ('gray', st['gray']), ('enhanced', st['enhanced'][..., ::-1]), ('mask', st['mask'])]
    for ax, (t, im) in zip(row, imgs):
        ax.imshow(im, cmap='gray' if im.ndim == 2 else None); ax.set_title(f"{r['class_name']} - {t}"); ax.axis('off')
plt.show()

## 3. Features

In [ ]:
res = [extract_from_path(p) for p in df['path']]
ok = [i for i, (v, e) in enumerate(res) if v is not None]
X = np.vstack([res[i][0] for i in ok]); df = df.iloc[ok].reset_index(drop=True); y = df['label'].to_numpy()
feat = pd.DataFrame(X, columns=FEATURE_NAMES); feat['label'] = y
feat.groupby('label').mean().T.head(20)

## 4. Split (same function as the real pipeline) and scaling fitted on train only

In [ ]:
train, val, test, mode = split_data(y, df['group'].to_numpy(), config.RANDOM_STATE)
print(mode, len(train), len(val), len(test))
scaler = StandardScaler().fit(X[train])
Xtr, Xva = scaler.transform(X[train]), scaler.transform(X[val])
ytr, yva = y[train], y[val]

## 5. Explore K and distance metric on VALIDATION data

In [ ]:
rows = []
for metric in config.DISTANCE_METRICS:
    for k in config.K_VALUES:
        if k > len(ytr): continue
        m = compute_metrics(yva, KNeighborsClassifier(n_neighbors=k, metric=metric).fit(Xtr, ytr).predict(Xva))
        rows.append({'k': k, 'metric': metric, 'balanced_accuracy': m['balanced_accuracy'], 'recall': m['recall'], 'specificity': m['specificity']})
res_df = pd.DataFrame(rows)
sns.lineplot(data=res_df, x='k', y='balanced_accuracy', hue='metric', marker='o'); plt.show()
res_df.sort_values('balanced_accuracy', ascending=False).head(10)

## 6. Next step
When satisfied, run `python -m src.train_model` from the project root. It does the final fit and the single test-set evaluation.